# 02 - Estimación del pulso a partir de un video real

**Qué hace este notebook:** carga un video con un rostro de frente, selecciona una región de interés (ROI), extrae las series temporales R, G, B, las caracteriza y estima el BPM con espectro de frecuencia.

**Cómo grabar un buen video de prueba**
- Celular apoyado (sin mano), rostro de frente y completo, 20 a 30 s, sin hablar ni moverse.
- Luz estable (luz natural o una lámpara fija); evita luz que parpadee y sombras cambiantes.
- Mide tu pulso a mano en el cuello durante 30 s (x2) para tener una referencia aproximada.
- Solo usa videos tuyos o de personas que dieron su consentimiento.

> Uso educativo. No es una medición clínica ni diagnóstica.

In [ ]:
import os, sys, subprocess

# --- Configuración (EDITA REPO_URL con el enlace de TU repositorio de GitHub) ---
REPO_URL = "https://github.com/sh-011188/Proyecto_Senales.git"
REPO_DIR = "Proyecto_Senales"   # nombre de la carpeta del repositorio

IN_COLAB = "google.colab" in sys.modules
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                           # ejecución local: subir a la raíz del repo
elif not os.path.isdir("src"):
    if "TU_USUARIO" in REPO_URL:
        raise SystemExit("Edita REPO_URL con el enlace de tu repositorio y vuelve a ejecutar.")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

sys.path.insert(0, os.getcwd())
print("Directorio de trabajo:", os.getcwd())


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.video_io import get_video_info
from src.pipeline import analyze_video

## 1. Cargar el video

In [ ]:
SUBIR_VIDEO = IN_COLAB                    # en Colab: aparece el botón para subir el archivo
VIDEO_PATH = "data/mi_video.mp4"          # ruta local (si el video ya está en data/)

if SUBIR_VIDEO:
    from google.colab import files
    subido = files.upload()
    VIDEO_PATH = next(iter(subido))
print("Video:", VIDEO_PATH)

In [ ]:
ROI_MODE = "frente"      # "frente", "mejillas" o "rostro"
MANUAL_ROI = None        # o (x, y, w, h) en píxeles, si el detector falla o la ROI sale mal
ESPERADO_BPM = None      # tu pulso medido a mano, p. ej. 72 (opcional pero recomendado)

info = get_video_info(VIDEO_PATH)
pd.Series({
    "fs = FPS (Hz)": info["fps"], "Ts = 1/fs (ms)": round(info["Ts_s"] * 1000, 2),
    "ancho x alto (px)": f"{info['width']} x {info['height']}",
    "fotogramas": info["n_frames_nominal"], "duración (s)": round(info["duration_s"], 1),
})

## 2. Análisis completo

Se detecta el rostro, se define la ROI, se extrae la serie RGB, se descartan los primeros segundos (auto-exposición), se procesa y se estima el BPM. **Revisa primero la figura de la ROI**: si toca cabello, cejas o fondo, cambia `ROI_MODE` o usa `MANUAL_ROI`.

In [ ]:
resultado = analyze_video(VIDEO_PATH, roi_mode=ROI_MODE, manual_roi=MANUAL_ROI,
                          expected_bpm=ESPERADO_BPM, out_dir="results", show=True, prefix="video_")

### ✍️ Interpretación (una por figura)
1. **ROI:** ¿cae sobre piel? ¿Qué se excluyó? ¿Por qué se eligió esa zona?
2. **Series RGB:** ¿qué nivel medio y qué deriva ves? ¿Se distingue el pulso a simple vista? ¿Hay saltos (exposición, movimiento)?
3. **Preprocesado:** ¿qué cambió al normalizar y quitar tendencia?
4. **Espectro (POS):** ¿dónde está el pico dominante en Hz y en BPM? ¿Hay otros picos (armónico, ruido)?
5. **Comparación:** ¿coincide con tu referencia manual? ¿Cuánto es el error y es mayor o menor que la resolución espectral?

*(Escribe aquí tus respuestas.)*

In [ ]:
pd.DataFrame(resultado["descripcion_canales"]).T[
    ["n_muestras", "fs_hz", "duracion_s", "minimo", "maximo", "amplitud_pico_a_pico", "media", "desv_estandar"]].round(3)

## 3. Comparar zonas del rostro (opcional)

Se repite el análisis con cada ROI. Si el pulso es consistente entre zonas, la estimación es más creíble.

In [ ]:
filas = []
for modo in ("frente", "mejillas", "rostro"):
    try:
        r = analyze_video(VIDEO_PATH, roi_mode=modo, expected_bpm=ESPERADO_BPM,
                          out_dir="results", show=False, prefix=f"video_{modo}_")
        filas.append({"ROI": modo, "BPM (POS)": round(r["bpm_pos"], 1), "calidad POS": round(r["quality_pos"], 2),
                      "BPM (verde)": round(r["bpm_verde"], 1)})
    except Exception as e:
        filas.append({"ROI": modo, "BPM (POS)": f"error: {e}"})
pd.DataFrame(filas)

## 4. Limitaciones observadas (escríbelas tú)
- ¿Qué pudo afectar el resultado (movimiento, luz, compresión del video, duración)?
- ¿Qué mejorarías en la entrega final (filtro pasa banda, ventanas, otra ROI)?

*(Escribe aquí.)*

In [ ]:
# Solo en Colab: descargar las figuras para subirlas a GitHub (carpeta results/)
if IN_COLAB:
    import shutil
    shutil.make_archive("results_figuras", "zip", "results")
    files.download("results_figuras.zip")